In [ ]:
# !pip install opencv-python numpy ultralytics deep-sort-realtime matplotlib

import cv2
import numpy as np
import os
import csv
import time
import matplotlib.pyplot as plt
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

# Display settings for notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]

In [ ]:
# Paths
INPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content"
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed"
VIDEO_NAME = "cctv052x2004080516x01640.avi" # Ensure this matches your specific file name in the folder
INPUT_VIDEO_PATH = os.path.join(INPUT_DIR, VIDEO_NAME)

# Make sure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Video Processing Constants
CONF_THRESHOLD = 0.50
TARGET_CLASS_ID = None # Set to 2 for cars if strictly filtering, None for all trained classes

# Perspective Transform Constants (From your provided script)
# Adjusted for what likely looks like the 320x240 settings in your script
FRAME_WIDTH_M = 30
FRAME_HEIGHT_M = 100

# Region of Interest (ROI) Points for Perspective Transform
SOURCE_POLYGON = np.array([[20, 200], [300, 220], [280, 100], [40, 80]], dtype=np.float32)
BIRD_EYE_VIEW = np.array([[0, 0], [FRAME_WIDTH_M, 0], [FRAME_WIDTH_M, FRAME_HEIGHT_M], [0, FRAME_HEIGHT_M]], dtype=np.float32)

TRANSFORM_MATRIX = cv2.getPerspectiveTransform(SOURCE_POLYGON, BIRD_EYE_VIEW)

In [ ]:
import cv2
import numpy as np
import os

# --- STEP 1: GENERATE DIFFICULT DATASETS ---
def generate_stress_dataset(input_video_path):
    print(f"Generating Stress Test Variations for: {input_video_path}...")
    cap = cv2.VideoCapture(input_video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    
    # Paths for new video files
    base = os.path.splitext(input_video_path)[0]
    path_dark = f"{base}_dark.mp4"
    path_washed = f"{base}_washed.mp4"
    
    out_dark = cv2.VideoWriter(path_dark, fourcc, fps, (width, height))
    out_washed = cv2.VideoWriter(path_washed, fourcc, fps, (width, height))
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        # Variation 1: Simulated Night/Underexposure (The "Baseline Killer")
        # Darkens image by 60 units. Baseline should fail to see dark cars.
        dark_frame = cv2.convertScaleAbs(frame, alpha=1.0, beta=-60)
        
        # Variation 2: Low Contrast / Glare
        # Smashes the histogram. Edges become fuzzy. Baseline confidence should drop.
        washed_frame = cv2.convertScaleAbs(frame, alpha=0.5, beta=40)
        
        out_dark.write(dark_frame)
        out_washed.write(washed_frame)
        
    cap.release()
    out_dark.release()
    out_washed.release()
    return [path_dark, path_washed]

# --- STEP 2: RUN THE PIPELINE ON THEM ---
# (Assumes you have your process_video function defined from previous steps)
stress_videos = generate_stress_dataset(INPUT_VIDEO_PATH)

# for video in stress_videos:
#     print(f"Processing {video}...")
#     process_video(video, method='none', output_filename=f"result_{os.path.basename(video)}_base.mp4")
#     process_video(video, method='clahe', output_filename=f"result_{os.path.basename(video)}_clahe.mp4")

In [ ]:
def show_frame(image, title="Frame", cmap=None):
    """Helper to display a single frame in the notebook."""
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

def show_comparison(img_original, img_processed, title_processed):
    """Side-by-side comparison of original and processed frames."""
    fig, ax = plt.subplots(1, 2, figsize=(20, 10))
    ax[0].imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Original")
    ax[0].axis('off')
    
    ax[1].imshow(cv2.cvtColor(img_processed, cv2.COLOR_BGR2RGB))
    ax[1].set_title(title_processed)
    ax[1].axis('off')
    plt.show()

# Load a sample frame to test our processing on
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
ret, sample_frame = cap.read()
cap.release()

if ret:
    show_frame(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB), title="Raw Sample Frame")
else:
    print("Error loading video. Check path.")

In [ ]:
def apply_he(frame):
    """
    Global Histogram Equalization.
    Applied to the Y channel of the YUV color space to preserve color information.
    """
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    # Equalize the histogram of the Y channel
    img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
    # Convert the YUV image back to RGB format
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_clahe(frame, clip_limit=2.0, tile_grid_size=(8,8)):
    """
    Contrast Limited Adaptive Histogram Equalization (CLAHE).
    Better for traffic video as it prevents noise amplification in uniform areas (road).
    """
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_gamma_correction(frame, gamma=1.2):
    """
    Gamma Correction.
    Non-linear operation to decode luminance. 
    Gamma > 1 makes image darker (good for washed out).
    Gamma < 1 makes image lighter (good for dark shadows).
    """
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
        for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(frame, table)

# --- VISUALIZE THE DIFFERENCES ---
if ret:
    he_frame = apply_he(sample_frame)
    clahe_frame = apply_clahe(sample_frame)
    gamma_frame = apply_gamma_correction(sample_frame, gamma=1.5) # Assuming washed out footage

    show_comparison(sample_frame, he_frame, "Global Histogram Equalization")
    show_comparison(sample_frame, clahe_frame, "CLAHE (Adaptive)")
    show_comparison(sample_frame, gamma_frame, "Gamma Correction")

In [ ]:
def calculate_distance(p1, p2):
    """Euclidean distance between two points."""
    return np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def calculate_speed(distance, fps):
    """
    Calculate speed based on distance and FPS.
    Returns speed in km/h.
    """
    return (distance * fps) * 3.6

def draw_corner_rect(img, bbox, line_length=30, line_thickness=5, rect_thickness=1,
                     rect_color=(255, 0, 255), line_color=(0, 255, 0)):
    x, y, w, h = bbox
    x1, y1 = x + w, y + h
    if rect_thickness != 0:
        cv2.rectangle(img, bbox, rect_color, rect_thickness)
    # Top Left
    cv2.line(img, (x, y), (x + line_length, y), line_color, line_thickness)
    cv2.line(img, (x, y), (x, y + line_length), line_color, line_thickness)
    # Top Right
    cv2.line(img, (x1, y), (x1 - line_length, y), line_color, line_thickness)
    cv2.line(img, (x1, y), (x1, y + line_length), line_color, line_thickness)
    # Bottom Left
    cv2.line(img, (x, y1), (x + line_length, y1), line_color, line_thickness)
    cv2.line(img, (x, y1), (x, y1 - line_length), line_color, line_thickness)
    # Bottom Right
    cv2.line(img, (x1, y1), (x1 - line_length, y1), line_color, line_thickness)
    cv2.line(img, (x1, y1), (x1, y1 - line_length), line_color, line_thickness)
    return img

In [ ]:
# def process_video(method='none', output_filename='output.mp4'):
#     print(f"Starting processing with method: {method}...")
    
#     cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
#     if not cap.isOpened():
#         print("Error opening video file")
#         return

#     # Video properties
#     frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     fps = int(cap.get(cv2.CAP_PROP_FPS))
    
#     # Setup Output Writer
#     output_path = os.path.join(OUTPUT_DIR, output_filename)
#     fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#     writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

#     # Initialize Models
#     tracker = DeepSort(max_age=50)
#     model = YOLO("yolov8n.pt") # Using YOLOv8 nano for speed, or use "yolov10n.pt" if you have it
    
#     # Class names (standard COCO)
#     model.model.names
    
#     # Tracking variables
#     prev_positions = {}
#     speed_accumulator = {}
#     frame_count = 0
    
#     # ROI Polygon for visualization
#     pts = SOURCE_POLYGON.astype(np.int32).reshape((-1, 1, 2))
#     polygon_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
#     cv2.fillPoly(polygon_mask, [pts], 255)

#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break
            
#         # 1. APPLY PRE-PROCESSING HERE
#         processed_frame = frame.copy()
#         if method == 'clahe':
#             processed_frame = apply_clahe(frame)
#         elif method == 'he':
#             processed_frame = apply_he(frame)
#         elif method == 'gamma':
#             processed_frame = apply_gamma_correction(frame)
        
#         # 2. DETECTION (Run YOLO on the PROCESSED frame)
#         results = model(processed_frame, verbose=False)
        
#         detect = []
#         for pred in results:
#             for box in pred.boxes:
#                 x1, y1, x2, y2 = map(int, box.xyxy[0])
#                 confidence = float(box.conf[0])
#                 label = int(box.cls[0])
                
#                 # Basic Filtering
#                 if confidence < CONF_THRESHOLD:
#                     continue
                
#                 # Check if center of box is in ROI
#                 center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
#                 if 0 <= center_y < frame_height and 0 <= center_x < frame_width:
#                      if polygon_mask[center_y, center_x] == 255:
#                         detect.append([[x1, y1, x2 - x1, y2 - y1], confidence, label])
        
#         # 3. TRACKING
#         tracks = tracker.update_tracks(detect, frame=processed_frame)
        
#         for track in tracks:
#             if not track.is_confirmed():
#                 continue
            
#             track_id = track.track_id
#             ltrb = track.to_ltrb()
#             x1, y1, x2, y2 = map(int, ltrb)
            
#             # Check ROI again for tracker
#             if polygon_mask[int((y1+y2)/2), int((x1+x2)/2)] == 0:
#                 continue
                
#             # 4. SPEED ESTIMATION
#             center_pt = np.array([[(x1+x2)//2, (y1+y2)//2]], dtype=np.float32)
#             transformed_pt = cv2.perspectiveTransform(center_pt[None, :, :], TRANSFORM_MATRIX)
            
#             current_speed = 0
#             if track_id in prev_positions:
#                 prev_pos = prev_positions[track_id]
#                 distance = calculate_distance(prev_pos, transformed_pt[0][0])
#                 current_speed = calculate_speed(distance, fps)
                
#                 # Smooth speed
#                 if track_id not in speed_accumulator:
#                     speed_accumulator[track_id] = []
#                 speed_accumulator[track_id].append(current_speed)
#                 if len(speed_accumulator[track_id]) > 5: # Moving average window
#                     speed_accumulator[track_id].pop(0)
                
#             prev_positions[track_id] = transformed_pt[0][0]
            
#             # 5. VISUALIZATION
#             avg_speed = 0
#             if track_id in speed_accumulator and speed_accumulator[track_id]:
#                 avg_speed = sum(speed_accumulator[track_id]) / len(speed_accumulator[track_id])

#             # Draw on original frame (or processed_frame if you prefer to see the effect in video)
#             draw_corner_rect(frame, (x1, y1, x2-x1, y2-y1))
            
#             label_text = f"ID:{track_id} {avg_speed:.1f} km/h"
#             cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

#         # Draw ROI polygon for reference
#         cv2.polylines(frame, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
        
#         writer.write(frame)
#         frame_count += 1
#         if frame_count % 50 == 0:
#             print(f"Processed {frame_count} frames...")

#     cap.release()
#     writer.release()
#     print(f"Done! Saved to {output_path}")

# # Run the pipeline
# # process_video(method='none', output_filename='baseline.mp4')
# # process_video(method='clahe', output_filename='clahe_enhanced.mp4')

import csv

def process_video(method='none', output_filename='output.mp4'):
    print(f"Starting processing with method: {method}...")
    
    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
    if not cap.isOpened():
        print("Error opening video file")
        return

    # Video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    # Setup Output Writer
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    # --- SETUP DATA LOGGER ---
    csv_filename = os.path.join(OUTPUT_DIR, f"data_log_{method}.csv")
    csv_file = open(csv_filename, 'w', newline='')
    csv_writer = csv.writer(csv_file)
    # Header: Frame Number, Track ID, Speed, Avg Frame Confidence (Quality Metric), Total Detections (Stability Metric)
    csv_writer.writerow(["Frame", "TrackID", "Speed_kmh", "Avg_Frame_Confidence", "Detections_In_Frame"])

    # Initialize Models
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt") 
    
    # Tracking variables
    prev_positions = {}
    speed_accumulator = {}
    frame_count = 0
    
    # ROI Polygon
    pts = SOURCE_POLYGON.astype(np.int32).reshape((-1, 1, 2))
    polygon_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
    cv2.fillPoly(polygon_mask, [pts], 255)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # 1. APPLY PRE-PROCESSING
        processed_frame = frame.copy()
        if method == 'clahe':
            processed_frame = apply_clahe(frame)
        elif method == 'he':
            processed_frame = apply_he(frame)
        elif method == 'gamma':
            processed_frame = apply_gamma_correction(frame)
        
        # 2. DETECTION
        results = model(processed_frame, verbose=False)
        
        detect = []
        confidences_in_this_frame = [] # List to store all confidence scores for this frame
        
        for pred in results:
            for box in pred.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = float(box.conf[0])
                label = int(box.cls[0])
                
                # Basic Filtering
                if confidence < CONF_THRESHOLD:
                    continue
                
                # Check ROI
                center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                if 0 <= center_y < frame_height and 0 <= center_x < frame_width:
                     if polygon_mask[center_y, center_x] == 255:
                        detect.append([[x1, y1, x2 - x1, y2 - y1], confidence, label])
                        confidences_in_this_frame.append(confidence)
        
        # Calculate Frame Metrics
        avg_frame_conf = 0
        if len(confidences_in_this_frame) > 0:
            avg_frame_conf = sum(confidences_in_this_frame) / len(confidences_in_this_frame)
        total_detections = len(detect)

        # 3. TRACKING
        tracks = tracker.update_tracks(detect, frame=processed_frame)
        
        for track in tracks:
            if not track.is_confirmed():
                continue
            
            track_id = track.track_id
            ltrb = track.to_ltrb()
            x1, y1, x2, y2 = map(int, ltrb)
            
            # Check ROI for tracker
            if polygon_mask[int((y1+y2)/2), int((x1+x2)/2)] == 0:
                continue
                
            # 4. SPEED ESTIMATION
            center_pt = np.array([[(x1+x2)//2, (y1+y2)//2]], dtype=np.float32)
            transformed_pt = cv2.perspectiveTransform(center_pt[None, :, :], TRANSFORM_MATRIX)
            
            current_speed = 0
            if track_id in prev_positions:
                prev_pos = prev_positions[track_id]
                distance = calculate_distance(prev_pos, transformed_pt[0][0])
                current_speed = calculate_speed(distance, fps)
                
                # Smooth speed
                if track_id not in speed_accumulator:
                    speed_accumulator[track_id] = []
                speed_accumulator[track_id].append(current_speed)
                if len(speed_accumulator[track_id]) > 5: 
                    speed_accumulator[track_id].pop(0)
                
            prev_positions[track_id] = transformed_pt[0][0]
            
            # 5. VISUALIZATION & LOGGING
            avg_speed = 0
            if track_id in speed_accumulator and speed_accumulator[track_id]:
                avg_speed = sum(speed_accumulator[track_id]) / len(speed_accumulator[track_id])

            # --- LOG DATA TO CSV ---
            # We log every confirmed track for every frame it appears in
            csv_writer.writerow([frame_count, track_id, avg_speed, avg_frame_conf, total_detections])

            # Draw visual elements
            draw_corner_rect(frame, (x1, y1, x2-x1, y2-y1))
            label_text = f"ID:{track_id} {avg_speed:.1f} km/h"
            cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Draw ROI polygon
        cv2.polylines(frame, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
        
        writer.write(frame)
        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    writer.release()
    csv_file.close() # Close the CSV file
    print(f"Done! Video saved to {output_path}")
    print(f"Data log saved to {csv_filename}")

In [ ]:
# 1. Run Baseline
process_video(method='none', output_filename='result_baseline.mp4')

# 2. Run CLAHE (Contrast Limited Adaptive Histogram Equalization)
# This is usually the best for traffic video as it handles shadows better than global HE
process_video(method='clahe', output_filename='result_clahe.mp4')

# 3. Run Global Histogram Equalization
process_video(method='he', output_filename='result_he.mp4')

In [ ]:
import cv2
import numpy as np
import os

def adjust_contrast_brightness(frame, alpha, beta):
    """
    alpha: Contrast control (1.0-3.0). <1.0 lowers contrast.
    beta: Brightness control (0-100).
    Formula: new_image = alpha * image + beta
    """
    new_frame = cv2.convertScaleAbs(frame, alpha=alpha, beta=beta)
    return new_frame

def generate_contrast_dataset(input_video_path):
    print(f"Creating Contrast Variations for: {input_video_path}...")
    
    cap = cv2.VideoCapture(input_video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    
    # Define Output Paths
    base_dir = os.path.dirname(input_video_path)
    base_name = os.path.splitext(os.path.basename(input_video_path))[0]
    
    path_dark = os.path.join(base_dir, f"{base_name}_dark.mp4")
    path_washed = os.path.join(base_dir, f"{base_name}_washed.mp4")
    
    out_dark = cv2.VideoWriter(path_dark, fourcc, fps, (width, height))
    out_washed = cv2.VideoWriter(path_washed, fourcc, fps, (width, height))
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # 1. Generate Underexposed (Dark)
        # Contrast = 1.0 (Same), Brightness = -60 (Darker)
        dark_frame = adjust_contrast_brightness(frame, alpha=1.0, beta=-60)
        
        # 2. Generate Low Contrast (Washed Out)
        # Contrast = 0.5 (Squished histogram), Brightness = +40 (Lift shadows)
        washed_frame = adjust_contrast_brightness(frame, alpha=0.5, beta=40)
        
        out_dark.write(dark_frame)
        out_washed.write(washed_frame)
        
    cap.release()
    out_dark.release()
    out_washed.release()
    
    print("Dataset Created!")
    print(f"1. Dark Video: {path_dark}")
    print(f"2. Washed Video: {path_washed}")
    return [input_video_path, path_dark, path_washed]

# --- EXECUTE ---
contrast_dataset = generate_contrast_dataset(INPUT_VIDEO_PATH)

In [ ]:
def run_contrast_stress_test(video_list):
    print("--- STARTING CONTRAST STRESS TEST ---")
    
    # We loop through every video condition (Normal, Dark, Washed)
    for video_path in video_list:
        condition_name = os.path.basename(video_path).split('.')[0] # e.g., "highway_dark"
        print(f"\nProcessing Condition: {condition_name}")
        
        # EXPERIMENT A: Baseline (No Processing)
        # We need to temporarily point the global INPUT_VIDEO_PATH to the current file
        # (Note: In a cleaner script we'd pass arguments, but this works with your current function)
        global INPUT_VIDEO_PATH 
        INPUT_VIDEO_PATH = video_path
        
        print(f"   > Running Baseline...")
        process_video(method='none', output_filename=f"result_{condition_name}_baseline.mp4")
        # Rename the CSV to avoid overwriting
        os.rename(os.path.join(OUTPUT_DIR, "data_log_none.csv"), 
                  os.path.join(OUTPUT_DIR, f"data_{condition_name}_baseline.csv"))
        
        # EXPERIMENT B: CLAHE (Our Solution)
        print(f"   > Running CLAHE...")
        process_video(method='clahe', output_filename=f"result_{condition_name}_clahe.mp4")
        # Rename the CSV
        os.rename(os.path.join(OUTPUT_DIR, "data_log_clahe.csv"), 
                  os.path.join(OUTPUT_DIR, f"data_{condition_name}_clahe.csv"))
                  
    print("\n--- ALL EXPERIMENTS DONE ---")
    print("Check your 'content' folder for 6 new CSV files.")

# --- EXECUTE ---
run_contrast_stress_test(contrast_dataset)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- SETUP ---
# Ensure your CSV files are in the same folder as this script
try:
    df_none = pd.read_csv('/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/data_highway_baseline.csv')
    df_he = pd.read_csv('/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/data_log_he.csv')
    df_clahe = pd.read_csv('/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/data_highway_clahe.csv')
except FileNotFoundError:
    print("Error: CSV files not found. Make sure you ran the batch experiment first!")
    # Create dummy data just so code runs if you are testing
    df_none = pd.DataFrame(columns=['Frame','TrackID','Speed_kmh','Avg_Frame_Confidence','Detections_In_Frame'])
    df_he = df_none.copy()
    df_clahe = df_none.copy()

# Label the DataFrames
df_none['Method'] = 'Baseline'
df_he['Method'] = 'Global HE'
df_clahe['Method'] = 'CLAHE'

# Combine into one big dataframe for plotting
df_all = pd.concat([df_none, df_he, df_clahe])

# --- METRIC CALCULATIONS ---
def calculate_metrics(df, method_name):
    if df.empty: return {}
    
    # 1. Stability: Count unique Track IDs
    unique_ids = df['TrackID'].nunique()
    
    # 2. Duration: Average lifespan of a track
    track_durations = df.groupby('TrackID')['Frame'].count()
    avg_duration = track_durations.mean() if not track_durations.empty else 0
    
    # 3. Confidence: Average confidence of all detections
    avg_conf = df['Avg_Frame_Confidence'].mean()
    
    # 4. Speed Noise: Average Std Dev of speed per vehicle
    # Only calculate for tracks that exist for > 5 frames
    long_tracks = track_durations[track_durations > 5].index
    if len(long_tracks) > 0:
        speed_std = df[df['TrackID'].isin(long_tracks)].groupby('TrackID')['Speed_kmh'].std()
        avg_noise = speed_std.mean()
    else:
        avg_noise = 0
        
    # 5. False Positives: Detections per frame
    avg_dets = df.groupby('Frame')['Detections_In_Frame'].mean().mean()
    
    return {
        'Method': method_name,
        'Unique IDs': unique_ids,
        'Avg Duration': avg_duration,
        'Avg Confidence': avg_conf,
        'Speed Noise': avg_noise,
        'Avg Detections': avg_dets
    }

# Generate the Table
metrics_list = [
    calculate_metrics(df_none, 'Baseline'),
    calculate_metrics(df_he, 'Global HE'),
    calculate_metrics(df_clahe, 'CLAHE')
]
results_df = pd.DataFrame(metrics_list).set_index('Method')

print("--- RESULTS TABLE FOR LATEX ---")
print(results_df)
print("-------------------------------")

# --- PLOTTING THE DASHBOARD ---
plt.style.use('seaborn-v0_8-whitegrid') # Or 'ggplot' if seaborn style fails
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3)

# 1. Stability Plot (Bar)
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x=results_df.index, y='Unique IDs', data=results_df, ax=ax1, palette=['gray', 'red', 'green'])
ax1.set_title('Tracking Stability (Lower is Better)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Total Unique IDs Created')
for container in ax1.containers:
    ax1.bar_label(container)

# 2. Confidence Plot (Bar)
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(x=results_df.index, y='Avg Confidence', data=results_df, ax=ax2, palette=['gray', 'red', 'green'])
ax2.set_title('Detection Confidence (Higher is Better)', fontsize=14, fontweight='bold')
ax2.set_ylim(0.5, 0.8) # Zoom in
for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f')

# 3. Speed Noise Plot (Bar)
ax3 = fig.add_subplot(gs[0, 2])
sns.barplot(x=results_df.index, y='Speed Noise', data=results_df, ax=ax3, palette=['gray', 'red', 'green'])
ax3.set_title('Speed Estimation Noise (Lower is Better)', fontsize=14, fontweight='bold')
ax3.set_ylabel('Avg Speed Std Dev (km/h)')
for container in ax3.containers:
    ax3.bar_label(container, fmt='%.2f')

# 4. Gantt Chart (Tracking Lifelines)
ax4 = fig.add_subplot(gs[1, :])

# Prepare Gantt Data
colors = {'Baseline': 'gray', 'Global HE': 'red', 'CLAHE': 'green'}
y_pos = 0
yticks = []
yticklabels = []

# Plot bars for each method
for method, df_curr in [('Baseline', df_none), ('Global HE', df_he), ('CLAHE', df_clahe)]:
    if df_curr.empty: continue
    
    # Get start/end for each track
    ranges = df_curr.groupby('TrackID')['Frame'].agg(['min', 'max']).sort_values('min')
    
    # Draw bars
    for i, (track_id, row) in enumerate(ranges.iterrows()):
        duration = row['max'] - row['min']
        ax4.barh(y_pos + i, duration, left=row['min'], height=0.8, 
                 color=colors[method], alpha=0.8, edgecolor='black', linewidth=0.5)
    
    # Label positioning
    mid_point = y_pos + len(ranges) / 2
    yticks.append(mid_point)
    yticklabels.append(method)
    
    y_pos += len(ranges) + 10 # Add visual gap between methods

ax4.set_yticks(yticks)
ax4.set_yticklabels(yticklabels, fontsize=12, fontweight='bold')
ax4.set_xlabel('Frame Number')
ax4.set_title('Vehicle Tracking Lifelines: Solid Bars = Stable Tracking', fontsize=14, fontweight='bold')
ax4.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('full_analytics_dashboard.png', dpi=300)
plt.show()
print("Graph saved as 'full_analytics_dashboard.png'")